# 2026 World Cup Prediction

We will run the model prediction for every single matches that will be played in the world cup to predict the outcome of each matches, as well as who will advance to the round of 32, 26, quarter finals, semi finals, and the finals.

We will get to see the result of the world cup!

In [1]:
!pip install ipynb

In [2]:
# Importing libraries
import numpy as np
import pandas as pd
import json

# Importing helper methods from the milestone notebook. These are for feature engineering.
from ipynb.fs.defs.Project_Milestone_clean import (
    compute_elo, compute_h2h, compute_past_game_counts,
    compute_last_n_form, compute_last_5_goals_scored
)

from datetime import datetime, timedelta
import random
from collections import defaultdict

In [ ]:
# Get the data
X_train = pd.read_csv("../data/X_train.csv")
Y_train = pd.read_csv("../data/Y_train.csv")

X_val = pd.read_csv("../data/X_val.csv")
Y_val = pd.read_csv("../data/Y_val.csv")

X_test = pd.read_csv("../data/X_test.csv")
Y_test = pd.read_csv("../data/Y_test.csv")

# Get the ML model of our choice

## This is the exact same Neural Network in our FFNN notebook.
Please refer to that notebook for more info.

In [4]:

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

import tensorflow as tf
from keras import models
from keras import layers

numeric_cols = ["elo_diff", "elo_sum", "home_team_h2h_wins", "away_team_h2h_wins", "h2h_draws", "rank_diff", "rank_sum", "home_count_last5", "away_count_last5", "pts_pg_diff_last5", "pts_pg_sum_last5", "gf_pg_diff_last5", "ga_pg_diff_last5"]
# binary_cols = ["neutral", "is_friendly"]

preprocessor = ColumnTransformer(
    transformers=[("standardized", StandardScaler(), numeric_cols)], remainder="passthrough")

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

tf.random.set_seed(1)
tf.keras.backend.clear_session()

model = tf.keras.Sequential()
model.add(tf.keras.Input(shape=(X_train_processed.shape[1],)))
model.add(tf.keras.layers.Dense(units=256, activation='relu'))
model.add(tf.keras.layers.Dense(units=128, activation='relu'))
model.add(tf.keras.layers.Dense(units=64, activation='relu'))
model.add(tf.keras.layers.Dense(units=3, activation='softmax'))

epochs = 20
learning_rate = 0.002

# Compile model
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

# Train model
history = model.fit(X_train_processed, Y_train,
                   validation_data=(X_val_processed, Y_val),
                   epochs=epochs, batch_size=64, verbose=1)



Epoch 1/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4895 - loss: 1.0423 - val_accuracy: 0.5189 - val_loss: 1.0090
Epoch 2/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5455 - loss: 0.9831 - val_accuracy: 0.5576 - val_loss: 0.9654
Epoch 3/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5669 - loss: 0.9488 - val_accuracy: 0.5774 - val_loss: 0.9388
Epoch 4/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5761 - loss: 0.9271 - val_accuracy: 0.5831 - val_loss: 0.9213
Epoch 5/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5796 - loss: 0.9125 - val_accuracy: 0.5860 - val_loss: 0.9090
Epoch 6/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5819 - loss: 0.9020 - val_accuracy: 0.5893 - val_loss: 0.9001
Epoch 7/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5841 - loss: 0.8943 - val_accuracy: 0.5918 - val_loss: 0.8933
Epoch 8/20
241/241 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5873 - loss: 0.8884 - val_accuracy: 0.

In [ ]:
# dataframe that contains the 2026 world cup matches for the group stages.
wc_matches = pd.read_csv("../data/wc_matches_with_features.csv")

In [6]:
# Map so that we can merge between the ranking df and the matches df with the uniform team name.
ranking_name_map = {
    "USA": "United States",
    "IR Iran": "Iran",
    "Türkiye": "Turkey",
    "Korea Republic": "South Korea",
    "Côte d'Ivoire": "Ivory Coast",
    "Congo DR": "DR Congo",
    "Cabo Verde": "Cape Verde",
    "Czechia": "Czech Republic"
}

In [ ]:
# Get the latest 2026 April FIFA rankings and generate a rank look up (team -> rank)
df_ranking_2026 = pd.read_csv("../data/2026_apr_ranking.csv")
df_ranking_2026["team"] = df_ranking_2026["team"].replace(ranking_name_map)
rank_lookup = df_ranking_2026.set_index("team")["rank"].to_dict()

In [8]:
# Helper function to build the feature matrix with the new matchups in the world cup.
# It will generate all the 15 features that we need, for the new match ups.

def build_feature_matrix(matchups_df, start_date, rank_lookup, feature_cols):

  # Get the historical matches
  df_history = pd.read_csv("./results.csv")
  base = datetime.strptime(start_date, "%Y-%m-%d")

  rows = []

  for i, (_, row) in enumerate(matchups_df.iterrows()):
    # For all the new matches, generate a new dataframe with the columns that are in-line with the original match data.

    rows.append({
      "date": (base + timedelta(days=i)).strftime("%Y-%m-%d"),
      "home_team": row["home_team"],
      "away_team": row["away_team"],
      "home_score": float("nan"), # we don't have a score yet.
      "away_score": float("nan"), # we don't have a score yet.
      "tournament": "FIFA World Cup",
      "neutral": True if ((row["home_team"] not in ["USA", "Canada", "Mexico"]) and (row["away_team"] not in ["USA", "Canada", "Mexico"])) else False
    })

  df = pd.concat([df_history, pd.DataFrame(rows)], ignore_index = True) # concaternate the historical matches with the new matches dataframe

  # Now compute all of the features just like before.

  df = compute_elo(df)
  df = compute_h2h(df)
  df = compute_past_game_counts(df)
  df = compute_last_n_form(df)

  df["normalized_points_home_team"] = np.where(df["home_count_last5"] == 0, 0, df["last_5_points_home_team"] / df["home_count_last5"])
  df["normalized_points_away_team"] = np.where(df["away_count_last5"] == 0, 0, df["last_5_points_away_team"] / df["away_count_last5"])
  df["pts_pg_diff_last5"] = df["normalized_points_home_team"] - df["normalized_points_away_team"]
  df["pts_pg_sum_last5"] = df["normalized_points_home_team"] + df["normalized_points_away_team"]

  df = compute_last_5_goals_scored(df)

  df["normalized_goals_for_home_team"] = np.where(df["home_count_last5"] == 0, 0, df["home_num_goals_scored_last_5_games"] / df["home_count_last5"])
  df["normalized_goals_for_away_team"] = np.where(df["away_count_last5"] == 0, 0, df["away_num_goals_scored_last_5_games"] / df["away_count_last5"])
  df["normalized_goals_against_home_team"] = np.where(df["home_count_last5"] == 0, 0, df["home_num_goals_conceded_last_5_games"] / df["home_count_last5"])
  df["normalized_goals_against_away_team"] = np.where(df["away_count_last5"] == 0, 0, df["away_num_goals_conceded_last_5_games"] / df["away_count_last5"])
  df["gf_pg_diff_last5"] = df["normalized_goals_for_home_team"] - df["normalized_goals_for_away_team"]
  df["ga_pg_diff_last5"] = df["normalized_goals_against_home_team"] - df["normalized_goals_against_away_team"]

  df["is_friendly"] = df["tournament"] == "Friendly"
  df["elo_diff"] = df["home_elo"] - df["away_elo"]
  df["elo_sum"] = df["home_elo"] + df["away_elo"]

  df["home_rank"] = df["home_team"].map(rank_lookup)
  df["away_rank"] = df["away_team"].map(rank_lookup)
  df["rank_diff"] = df["home_rank"] - df["away_rank"]
  df["rank_sum"] = df["home_rank"] + df["away_rank"]

  # Get the portion of the processed feature dataframe that corresponds to the world cup matches that we are interested in.
  wc = df[(df["tournament"] == "FIFA World Cup") & (df["date"] >= start_date)].copy()

  return wc, wc[feature_cols]

In [9]:
# This is a helper function that will allow us to get the team name that has won that particular matchup, given as a matchup df and a index to look for.
# Note that since this is a knock out stage, no draws are allowed.
# So if winning_class == 0 has the highest probability, then we'll select the next highest class to determine the knockout winner.

def get_knockout_winner(predictions, matchups_df, idx):
  row = matchups_df.iloc[idx]
  winning_class = np.argmax(predictions[idx])

  if (winning_class == 1):
    return row["home_team"]
  elif (winning_class == 2):
    return row["away_team"]
  else:
    # Class when they have predicted draw, but draw is not possible in knockout match.
    # Select the next highest class for the winner.
    return row["home_team"] if predictions[idx][1] >= predictions[idx][2] else row["away_team"]


In [10]:
# This is a helper method that will predict the outcome of the matches,
# and it will return us the maps of the winner and loser of that particular unique match number.

def predict_knockout_round(model, preprocessor, X_features, matchups_df):
  X_proc = preprocessor.transform(X_features)
  preds = model.predict(X_proc) # predict the outcome

  winners = {}
  losers = {}

  for i, (_, row) in enumerate(matchups_df.iterrows()):
    winner = get_knockout_winner(preds, matchups_df, i) # get the knockout winner
    loser = row["away_team"] if winner == row["home_team"] else row["home_team"]
    winners[row["match_num"]] = winner
    losers[row["match_num"]] = loser

  # return the original prediction array, as well as the winners and the losers of each matchups.

  return preds, winners, losers

In [11]:
# This is the helper method to build the matchups dataframe given a very raw form of the schedule of the world cup,
# and the function to resolve any unique strings inside the raw schedule dictionary to parse out the correct information.

def build_matchups_df(schedule_dict, resolve_fn):
  rows = []
  for num, val in schedule_dict.items():
    rows.append({
      "match_num": num,
      "home_team": resolve_fn(val["home"]), # get the home team from the raw string
      "away_team": resolve_fn(val["away"]) # get the away team from the raw string.
    })
  return pd.DataFrame(rows)

In [12]:
# Helper function to parse out the winning team by parsing out the match number from a raw string description of the match.

def get_winners_slot(slot_str, winners):
  return winners[int(slot_str.replace("Winner match ", ""))]

# Group Stage

In [13]:
# The groups in the 2026 FIFA World Cup, and the countries in each groups.
groups = {
  "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
  "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
  "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
  "D": ["United States", "Paraguay", "Australia", "Turkey"],
  "E": ["Germany", "Ivory Coast", "Ecuador", "Curaçao"],
  "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
  "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
  "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
  "I": ["France", "Senegal", "Iraq", "Norway"],
  "J": ["Argentina", "Algeria", "Austria", "Jordan"],
  "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
  "L": ["England", "Croatia", "Ghana", "Panama"]
}

In [14]:
X_wc2026 = wc_matches[X_train.columns]
predictions = model.predict(preprocessor.transform(X_wc2026)) # Predict the outcome of each group stage matches

# Group stages are unlike the knockout matches:
# It follows the 3/1/0 point system, where if they win, the winner get 3 points,
# if they draw, they each get 1 point, and if they lose, the loser gets 0 point.

# And after everyone plays their matches, the 2 teams with the highest points from each groups advances to the knockout rounds.
# And this year, we'll be selecting the best 3rd place teams across all the groups to advance to the next round.

# We'll mimic this in our project.


group_stage_points = defaultdict(int)
wc_matches_reset = wc_matches.reset_index(drop=True)

for i, row in wc_matches_reset.iterrows():
  winning_class = np.argmax(predictions[i]) # Get the outcome of the match.

  if (winning_class == 1): # Home team has won
    winning_team = row["home_team"]
    group_stage_points[row["home_team"]] += 3 # Add three points to home_team in the tracker
  elif (winning_class == 2): # Away team has won
    winning_team = row["away_team"]
    group_stage_points[row["away_team"]] += 3 # Add three points to away_team in the tracker
  else: # They have tied
    # Add one points to each
    group_stage_points[row["home_team"]] += 1
    group_stage_points[row["away_team"]] += 1


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


In [15]:
group_winners = {}

# Figure out the 1st and 2nd places from each groups.

for group_name, teams in groups.items():

  # Get a list of tuples (0th position being the team name and the 1st position being the total points) in a desc sorted way based on the points
  team_points = sorted([(t, group_stage_points.get(t, 0)) for t in teams], key=lambda x:x[1], reverse=True)

  # Get the 1st and 2nd places from each group.
  group_winners[group_name] = {
      "1st": {"team": team_points[0][0], "points": team_points[0][1]},
      "2nd": {"team": team_points[1][0], "points": team_points[1][1]}
      }


In [16]:
third_place_teams = []
# We'll now get the third place teams from each groups.
for group_name, teams in groups.items():

  # Get a list of tuples (0th position being the team name and the 1st position being the total points) in a desc sorted way based on the points
  team_points = sorted([(t, group_stage_points.get(t, 0)) for t in teams], key=lambda x:x[1], reverse=True)

  # Get the third place team and their points.
  third_team, third_pts = team_points[2]
  third_place_teams.append({
      "group": group_name,
      "team": third_team,
      "points": third_pts
      })

In [17]:
# Sort the third place team based on their points, and if their points are the same, just choose the team randomly to break ties.
third_place_teams.sort(key=lambda x: (x["points"], random.random()), reverse=True)

# Get the best 8 teams
best_third_place_teams = third_place_teams[:8]


In [18]:
round_of_32_teams = []

# Get the teams that are going to the round of 32 from the group stages.
for group_name, winners in group_winners.items():
  round_of_32_teams.append(winners["1st"]["team"])
  round_of_32_teams.append(winners["2nd"]["team"])


for item in best_third_place_teams:
  round_of_32_teams.append(item["team"])


# Round of 32

In [19]:
# round of 32 raw schedule.
# The key is the match number.

r32_schedule_raw = {
  73: {"home": "Group A runners-up", "away": "Group B runners-up"},
  74: {"home": "Group E winners", "away": "Group A/B/C/D/F third place"},
  75: {"home": "Group F winners", "away": "Group C runners-up"},
  76: {"home": "Group C winners", "away": "Group F runners-up"},
  77: {"home": "Group I winners", "away": "Group C/D/F/G/H third place"},
  78: {"home": "Group E runners-up", "away": "Group I runners-up"},
  79: {"home": "Group A winners", "away": "Group C/E/F/H/I third place"},
  80: {"home": "Group L winners", "away": "Group E/H/I/J/K third place"},
  81: {"home": "Group D winners", "away": "Group B/E/F/I/J third place"},
  82: {"home": "Group G winners", "away": "Group A/E/H/I/J third place"},
  83: {"home": "Group K runners-up", "away": "Group L runners-up"},
  84: {"home": "Group H winners", "away": "Group J runners-up"},
  85: {"home": "Group B winners", "away": "Group E/F/G/I/J third place"},
  86: {"home": "Group J winners", "away": "Group H runners-up"},
  87: {"home": "Group K winners", "away": "Group D/E/I/J/L third place"},
  88: {"home": "Group D runners-up", "away": "Group G runners-up"}
  }

In [20]:
position_lookup = {}
# Making a map for us to look up the 1st and 2nd group winners easily, given a raw string.

for group, info in group_winners.items():
  position_lookup[f"Group {group} winners"] = info["1st"]["team"]
  position_lookup[f"Group {group} runners-up"] = info["2nd"]["team"]


In [21]:

used_third_place = set()

# Helper method to get the specific team detail based on the raw match description

def resolve_r32_match_desc(match_desc):

  # If the match desc includes 1st or 2nd winner of the group, just look up on the position_lookup
  if (match_desc in position_lookup):
    return position_lookup[match_desc]

  # If the match desc includes third place
  if "third place" in match_desc:

    # Parse out the group names where the third place team should be parsed out from.
    eligible_groups = match_desc.replace("Group ", "").replace(" third place", "").split("/")

    for team in best_third_place_teams:

      # If the third place team is among the eligible groups and if we haven't yet placed this team in the tournament, then use this team.
      if team["group"] in eligible_groups and team["team"] not in used_third_place:
        used_third_place.add(team["team"])
        return team["team"]

    # just for safety, if we have missed out on mapping, then use the next unused third place team for the tournament matchup.
    for team in best_third_place_teams:
      if team["team"] not in used_third_place:
        used_third_place.add(team["team"])
        return team["team"]



In [22]:
# Get the round of 32 matchup
r32_matchups_df = build_matchups_df(r32_schedule_raw, resolve_r32_match_desc)

# Get the feature engineered for the round of 32 matchups
_, X_wc_r32 = build_feature_matrix(r32_matchups_df, "2026-06-28", rank_lookup, X_train.columns)


In [23]:
# Get the predictions of the round of 32.
r_32_predictions, r32_winners, _ = predict_knockout_round(model, preprocessor, X_wc_r32, r32_matchups_df)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


# Round of 16

In [24]:
# Raw round of 16 schedule.
# Match number is the key of this map.
r16_schedule_raw = {
  89: {"home": "Winner match 74", "away": "Winner match 77"},
  90: {"home": "Winner match 73", "away": "Winner match 75"},
  91: {"home": "Winner match 76", "away": "Winner match 78"},
  92: {"home": "Winner match 79", "away": "Winner match 80"},
  93: {"home": "Winner match 83", "away": "Winner match 84"},
  94: {"home": "Winner match 81", "away": "Winner match 82"},
  95: {"home": "Winner match 86", "away": "Winner match 88"},
  96: {"home": "Winner match 85", "away": "Winner match 87"}
  }

In [25]:
# Get the match up dataframe for round of 16.
# That lambda function is for parseing out the winner info given a raw match string.
r16_matchups_df = build_matchups_df(r16_schedule_raw, lambda s: get_winners_slot(s, r32_winners))

# generate the features
_, X_wc_r16 = build_feature_matrix(r16_matchups_df, "2026-07-04", rank_lookup, X_train.columns)



In [26]:
# Get the round of 16 predictions.

r16_predictions, r16_winners, _= predict_knockout_round(model, preprocessor, X_wc_r16, r16_matchups_df)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


# Quarter Finals

In [27]:
# Raw quarter finals schedule.
# Match number is the key.

qf_schedule_raw = {
    97:  {"home": "Winner match 89", "away": "Winner match 90"},
    98:  {"home": "Winner match 93", "away": "Winner match 94"},
    99:  {"home": "Winner match 91", "away": "Winner match 92"},
    100: {"home": "Winner match 95", "away": "Winner match 96"},
}

In [28]:
# Get the quarter finals dataframe.
qf_matchups_df = build_matchups_df(qf_schedule_raw, lambda s: get_winners_slot(s, r16_winners))

# Get the features set
_, X_wc_qf = build_feature_matrix(qf_matchups_df, "2026-07-09", rank_lookup, X_train.columns)


In [29]:
# Get the prediction of the quarter finals.
predictions_qf, qf_winners, _ = predict_knockout_round(model, preprocessor, X_wc_qf, qf_matchups_df)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


# Semi finals

In [30]:
# Raw semi finals schedule.
# Match number is the key.

sf_schedule_raw = {
    101: {"home": "Winner match 97",  "away": "Winner match 98"},
    102: {"home": "Winner match 99",  "away": "Winner match 100"},
}


In [31]:
# Get the semi finals dataframe.
sf_matchups_df = build_matchups_df(sf_schedule_raw, lambda s: get_winners_slot(s, qf_winners))

# Get the features set
_, X_wc_sf = build_feature_matrix(sf_matchups_df, "2026-07-14", rank_lookup, X_train.columns)

In [32]:
# Get the prediction of the semi finals.

predictions_sf, sf_winners, sf_losers = predict_knockout_round(model, preprocessor, X_wc_sf, sf_matchups_df)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


# finals

In [33]:
# Helper function to get the team/match up details based on the schedule string and the semi finals result
def resolve_finals_slot(slot_str):
    if "Winner match" in slot_str:
        return sf_winners[int(slot_str.replace("Winner match ", ""))]
    return sf_losers[int(slot_str.replace("Runner-up match ", ""))]

In [34]:
# Raw finals schedule.
# Match number is the key.

finals_schedule_raw = {
    103: {"home": "Runner-up match 101", "away": "Runner-up match 102"},
    104: {"home": "Winner match 101", "away": "Winner match 102"},
}

In [35]:
# Get the finals dataframe.
finals_matchups_df = build_matchups_df(finals_schedule_raw, resolve_finals_slot)

# Get the features set
_, X_wc_finals = build_feature_matrix(finals_matchups_df, "2026-07-19", rank_lookup, X_train.columns)

In [41]:
# Get the prediction of the finals.
predictions_finals, finals_winners, finals_losers = predict_knockout_round(model, preprocessor, X_wc_finals, finals_matchups_df)

champ = finals_winners[104]
print(f"FIFA World Cup 2026 winner: {champ}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
FIFA World Cup 2026 winner: France


In [43]:
def print_bracket():
  def collect(predictions, matchups_df):
    out = []
    for i, (_, row) in enumerate(matchups_df.iterrows()):
      winner = get_knockout_winner(predictions, matchups_df, i)
      out.append({
        "match": row["match_num"],
        "home": row["home_team"],
        "away": row["away_team"],
        "winner": winner,
        "home_prob": predictions[i][1],
        "away_prob": predictions[i][2]
        })
    return out

  rounds = [
    ("ROUND OF 32", collect(r_32_predictions, r32_matchups_df)),
    ("ROUND OF 16", collect(r16_predictions, r16_matchups_df)),
    ("QUARTERFINALS", collect(predictions_qf, qf_matchups_df)),
    ("SEMIFINALS", collect(predictions_sf, sf_matchups_df)),
    ("FINALS", collect(predictions_finals, finals_matchups_df))
    ]

  print("FIFA WORLD CUP 2026 - PREDICTIONS")
  print()

  for round_name, matches in rounds:
    print(round_name)
    for m in matches:
      h_tag = "WIN" if m["winner"] == m["home"] else ""
      a_tag = "WIN" if m["winner"] == m["away"] else ""
      h_pct = f"{m["home_prob"]*100:.1f}%"
      a_pct = f"{m["away_prob"]*100:.1f}%"
      print(f"{m["home"]} {h_pct} {h_tag}")
      print(f"{m["away"]} {a_pct} {a_tag}")
      print()

  champion = finals_winners[104]
  bronze_winner = finals_winners[103]

  print(f"WORLD CHAMPION: {champion}")
  print(f"Runner up: {finals_losers[104]}")

print_bracket()

FIFA WORLD CUP 2026 - PREDICTIONS

ROUND OF 32
South Korea 35.0% 
Switzerland 36.0% WIN

Germany 73.4% WIN
Bosnia and Herzegovina 8.3% 

Netherlands 40.6% WIN
Morocco 29.4% 

Brazil 45.4% WIN
Japan 23.7% 

France 76.2% WIN
Saudi Arabia 7.0% 

Ecuador 32.8% 
Senegal 37.3% WIN

Mexico 47.0% WIN
Norway 22.5% 

England 64.8% WIN
Ivory Coast 11.8% 

United States 41.4% WIN
Austria 28.0% 

Belgium 53.5% WIN
Scotland 17.5% 

Colombia 33.8% 
Croatia 36.0% WIN

Spain 63.9% WIN
Algeria 13.1% 

Canada 39.6% WIN
Egypt 30.0% 

Argentina 49.4% WIN
Uruguay 19.1% 

Portugal 61.3% WIN
Sweden 14.1% 

Australia 38.1% WIN
Iran 32.1% 

ROUND OF 16
Germany 29.6% 
France 39.7% WIN

Switzerland 32.0% 
Netherlands 39.0% WIN

Brazil 45.7% WIN
Senegal 24.5% 

Mexico 29.2% 
England 38.9% WIN

Croatia 27.4% 
Spain 41.6% WIN

United States 27.5% 
Belgium 44.5% WIN

Argentina 59.6% WIN
Australia 16.1% 

Canada 24.0% 
Portugal 44.3% WIN

QUARTERFINALS
France 42.5% WIN
Netherlands 27.1% 

Spain 46.0% WIN
Belgium 25.5%

In [38]:
print(json.dumps(group_winners, indent=4))


{
    "A": {
        "1st": {
            "team": "Mexico",
            "points": 9
        },
        "2nd": {
            "team": "South Korea",
            "points": 6
        }
    },
    "B": {
        "1st": {
            "team": "Canada",
            "points": 9
        },
        "2nd": {
            "team": "Switzerland",
            "points": 6
        }
    },
    "C": {
        "1st": {
            "team": "Brazil",
            "points": 9
        },
        "2nd": {
            "team": "Morocco",
            "points": 6
        }
    },
    "D": {
        "1st": {
            "team": "United States",
            "points": 9
        },
        "2nd": {
            "team": "Australia",
            "points": 6
        }
    },
    "E": {
        "1st": {
            "team": "Germany",
            "points": 9
        },
        "2nd": {
            "team": "Ecuador",
            "points": 6
        }
    },
    "F": {
        "1st": {
            "team": "Netherlands",
       

In [39]:
print(best_third_place_teams)

[{'group': 'I', 'team': 'Norway', 'points': 3}, {'group': 'H', 'team': 'Saudi Arabia', 'points': 3}, {'group': 'E', 'team': 'Ivory Coast', 'points': 3}, {'group': 'B', 'team': 'Bosnia and Herzegovina', 'points': 3}, {'group': 'C', 'team': 'Scotland', 'points': 3}, {'group': 'J', 'team': 'Austria', 'points': 3}, {'group': 'G', 'team': 'Egypt', 'points': 3}, {'group': 'F', 'team': 'Sweden', 'points': 3}]
